<a href="https://colab.research.google.com/github/jglim531/math-103.1-nlp/blob/main/03_Machine_Learning_Models_For_Text/Naive_Bayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Import Libraries

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
from collections import Counter

from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB

import pandas as pd

### Worked Example

In [ ]:
training_set = [
    "Pingo in Antarctica by Hale is dull and slow",
    "Pingo in Antarctica by Hale is a boring film",
    "Pingo in Antarctica by Hale is dull and predictable",
    "A funny film",
    "Charming and beautiful"
]

labels = [
    "Negative",
    "Negative",
    "Negative",
    "Positive",
    "Positive"
]

test_set = ["Pingo in Antarctica by Hale is charming and funny"]

#### Pre-processing Version 1
- lowercase
- remove name
- remove stopwords (spaCy)

In [ ]:
# Pre-process the training set
custom_names = ["Pingo in Antarctica", "Hale"]
custom_cleaned_training = []
custom_cleaned_test = []

def custom_preprocess(sentence):
    for name in custom_names:
        if name in sentence:
            sentence = sentence.replace(name, "")

    # Strip trailing spaces and double spaces
    sentence = sentence.strip()
    sentence = " ".join(sentence.split())

    return sentence


# Clean training set by removing custom names
for sentence in training_set:
    sentence = custom_preprocess(sentence)
    custom_cleaned_training.append(sentence)

# Clean testing set by removing custom names
for sentence in test_set:
    sentence = custom_preprocess(sentence)
    custom_cleaned_test.append(sentence)

In [ ]:
print(f"Custom Cleaned Training Set:")
print(custom_cleaned_training)
print()
print(f"Custom Cleaned Test Set:")
print(custom_cleaned_test)

Custom Cleaned Training Set:
['by is dull and slow', 'by is a boring film', 'by is dull and predictable', 'A funny film', 'Charming and beautiful']

Custom Cleaned Test Set:
['by is charming and funny']


In [ ]:
cleaned_training = []
cleaned_test = []

def spacy_preprocess(sentence):
    doc = nlp(sentence)

    tokens = []

    for token in doc:
        if token.ent_type_ != "PERSON" and not token.is_stop:  # Remove other names not in custom names and stopwords
            token_to_be_added = token.text.lower()
            tokens.append(token_to_be_added)

    # Show the tokens
    print(tokens)

    # Join the tokens together
    cleaned_text = " ".join(tokens)
    print(cleaned_text)

    return cleaned_text

# Clean by removing other names and stopwords
for sentence in custom_cleaned_training:
    final_training_text = spacy_preprocess(sentence)
    cleaned_training.append(final_training_text)

for sentence in custom_cleaned_test:
    final_test_text = spacy_preprocess(sentence)
    cleaned_test.append(final_test_text)

['dull', 'slow']
dull slow
['boring', 'film']
boring film
['dull', 'predictable']
dull predictable
['funny', 'film']
funny film
['charming', 'beautiful']
charming beautiful
['charming', 'funny']
charming funny


In [ ]:
print(f"Cleaned Training Set:")
print(cleaned_training)
print()
print(f"Cleaned Test Set:")
print(cleaned_test)

Cleaned Training Set:
['dull slow', 'boring film', 'dull predictable', 'funny film', 'charming beautiful']

Cleaned Test Set:
['charming funny']


#### Text Representation
- Use CountVectorizer for BoW (no need to do manually)
- Naive Bayes must be trained on the **training data** only

In [ ]:
# Load the training data
training_data = cleaned_training

# Create the CountVectorizer (this builds the BoW model)
# default is binary = False, so it counts the number of times a token appears in a sentence
# set binary = True, if we just want to check the presence of a token in a sentence
vectorizer = CountVectorizer(binary = False)

# Fit and transform the corpus into a BoW matrix
bow_representation = vectorizer.fit_transform(training_data)

print(f"Bag of Words Representation:\n")
print(bow_representation)

Bag of Words Representation:

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 10 stored elements and shape (5, 8)>
  Coords	Values
  (0, 3)	1
  (0, 7)	1
  (1, 1)	1
  (1, 4)	1
  (2, 3)	1
  (2, 6)	1
  (3, 4)	1
  (3, 5)	1
  (4, 2)	1
  (4, 0)	1


In [ ]:
# Make the bag of words vectors
print(f"Vocabulary: {vectorizer.get_feature_names_out()}\n")

for i in range(len(training_data)):
    print(f"Training Sentence {i + 1} ({labels[i]}): {training_data[i]}")
    print(f"BOW Representation {i + 1}: {bow_representation[i].toarray()}")
    print("")

Vocabulary: ['beautiful' 'boring' 'charming' 'dull' 'film' 'funny' 'predictable'
 'slow']

Training Sentence 1 (Negative): dull slow
BOW Representation 1: [[0 0 0 1 0 0 0 1]]

Training Sentence 2 (Negative): boring film
BOW Representation 2: [[0 1 0 0 1 0 0 0]]

Training Sentence 3 (Negative): dull predictable
BOW Representation 3: [[0 0 0 1 0 0 1 0]]

Training Sentence 4 (Positive): funny film
BOW Representation 4: [[0 0 0 0 1 1 0 0]]

Training Sentence 5 (Positive): charming beautiful
BOW Representation 5: [[1 0 1 0 0 0 0 0]]



#### Naive Bayes Classifier

The predicted class $\hat{c}$ is the class with the highest log prior plus the sum of the log likelihoods of the words in the test document:

$$
\hat{c} = \underset{c \in C}{\operatorname{argmax}} \; \Big[ \log P(c) + \sum_{i=1}^{n} \log P(w_i \mid c) \Big]
$$

where $w_1, \dots, w_n$ are the words of the test document that appear in the vocabulary $V$.

#### Parameter Estimates

**Prior**: the fraction of training documents that belong to class $c$

$$
P(c) = \frac{N_c}{N_{doc}}
$$

**Likelihood**: how often word $w_i$ appears in class $c$, relative to all words in class $c$, using Laplace (add-one) smoothing

$$
P(w_i \mid c) = \frac{\mathrm{count}(w_i, c) + 1}{\sum_{w \in V} \mathrm{count}(w, c) + |V|}
$$

In [ ]:
# This is how the vectorizer further pre-processes the already pre-processed text
# Make sure to keep this so we get consistent BoW representations
tokenize = vectorizer.build_analyzer()

def train_naive_bayes(D, labels):
    """
    D = documents or the training data (must be pre-processed based on our manual pre-processing)
    labels - class labels
    """
    # Gets the unique classes
    C = sorted(set(labels))

    # Log prior to calculate the predicted class
    # Dictionary that maps each class label to its log prior: log_prior[c] = log P(c)
    log_prior = {}

    # Log likelihood to calculate the predicted class
    # Dictionary that maps each (word, class) pair to its log likelihood: log_likelihood[(w, c)] = log P(w | c)
    log_likelihood = {}

    # Calculate P(c) terms
    N_doc = len(D)

    # V: vocabulary of D (manually construct the vocabulary from the documents)
    V = set() # Creates an empty set
    for d in D:
        V.update(tokenize(d)) # Adds an item to the set
    V = sorted(V)

    # Can also use CountVectorizer's get_feature_names_out() since we used it earlier
    # V = list(vectorizer.get_feature_names_out())

    # Loop through all classes
    for c in C:

        # Count how many documents belong to each class
        # Calculate the log_prior of each class
        N_c = sum(1 for label in labels if label == c)
        log_prior[c] = np.log(N_c / N_doc)

        # bigdoc is the collection of documents that belong to class c
        bigdoc = []

        # Combine into (document, label) to loop through it to know which document is part of bigdoc
        for d, label in zip(D, labels):
            if label == c:
                bigdoc.extend(tokenize(d))

        # Counts the number of times each word in the documents that belong to class c occurs
        counts = Counter(bigdoc)

        # Denominator with Laplace smoothing as derived
        denom = sum(counts[w] for w in V) + len(V)

        # Calculate P(w|c) terms
        for w in V:
            log_likelihood[(w, c)] = np.log((counts[w] + 1) / denom) # with Laplace smoothing

    return log_prior, log_likelihood, V, C


def test_naive_bayes(testdoc, log_prior, log_likelihood, C, V):
    # Reconstruct the testing vocabulary
    V = set(V)
    total = {}

    # Calculate the probability score for each classifier
    for c in C:
        total[c] = log_prior[c]
        for word in tokenize(testdoc):          # Tokenizes the words in test data similar to how it was done in training data
            if word in V:                       # Unknown words are skipped (as discussed)
                total[c] += log_likelihood[(word, c)] # Add the calculated likelihood that the word belongs to a certain class
    return max(total, key = total.get), total     # This is just argmax the resulting sum, max returns the key value of the total dictionary, which is the class, key is how you compare values

In [ ]:
# Perform the training step on our training data
log_prior, log_likelihood, V, C = train_naive_bayes(cleaned_training, labels)

print(f"Log Prior: {log_prior}\n")
print(f"Log Likelihood: {log_likelihood}\n")
print(f"Vocabulary: {V}\n")
print(f"Classes: {C}")

Log Prior: {'Negative': np.float64(-0.5108256237659907), 'Positive': np.float64(-0.916290731874155)}

Log Likelihood: {('beautiful', 'Negative'): np.float64(-2.639057329615259), ('boring', 'Negative'): np.float64(-1.9459101490553135), ('charming', 'Negative'): np.float64(-2.639057329615259), ('dull', 'Negative'): np.float64(-1.540445040947149), ('film', 'Negative'): np.float64(-1.9459101490553135), ('funny', 'Negative'): np.float64(-2.639057329615259), ('predictable', 'Negative'): np.float64(-1.9459101490553135), ('slow', 'Negative'): np.float64(-1.9459101490553135), ('beautiful', 'Positive'): np.float64(-1.791759469228055), ('boring', 'Positive'): np.float64(-2.4849066497880004), ('charming', 'Positive'): np.float64(-1.791759469228055), ('dull', 'Positive'): np.float64(-2.4849066497880004), ('film', 'Positive'): np.float64(-1.791759469228055), ('funny', 'Positive'): np.float64(-1.791759469228055), ('predictable', 'Positive'): np.float64(-2.4849066497880004), ('slow', 'Positive'): np.f

In [ ]:
argmax, total = test_naive_bayes(cleaned_test[0], log_prior, log_likelihood, C, V)

print(f"Cleaned Test Sentence: {cleaned_test[0]}")
print(f"Predicted Class: {argmax}")
print(f"Total: {total}")

Cleaned Test Sentence: charming funny
Predicted Class: Positive
Total: {'Negative': np.float64(-5.788940282996508), 'Positive': np.float64(-4.499809670330265)}


#### Pre-processing Version 2
- lowercase
- remove stopwords (spaCy)

In [ ]:
# Pre-Process
cleaned_training_v2 = []
cleaned_test_v2 = []

def spacy_preprocess(sentence):
    doc = nlp(sentence)

    tokens = []

    for token in doc:
        if not token.is_stop:  # Remove stopwords
            token_to_be_added = token.text.lower()
            tokens.append(token_to_be_added)

    # Show the tokens
    print(tokens)

    # Join the tokens together
    cleaned_text = " ".join(tokens)
    print(cleaned_text)

    return cleaned_text

# Clean by removing other names and stopwords
for sentence in training_set:
    final_training_text = spacy_preprocess(sentence)
    cleaned_training_v2.append(final_training_text)

for sentence in test_set:
    final_test_text = spacy_preprocess(sentence)
    cleaned_test_v2.append(final_test_text)

['pingo', 'antarctica', 'hale', 'dull', 'slow']
pingo antarctica hale dull slow
['pingo', 'antarctica', 'hale', 'boring', 'film']
pingo antarctica hale boring film
['pingo', 'antarctica', 'hale', 'dull', 'predictable']
pingo antarctica hale dull predictable
['funny', 'film']
funny film
['charming', 'beautiful']
charming beautiful
['pingo', 'antarctica', 'hale', 'charming', 'funny']
pingo antarctica hale charming funny


In [ ]:
print(f"Cleaned Training Set V2:")
print(cleaned_training_v2)
print()
print(f"Cleaned Test Set V2:")
print(cleaned_test_v2)

Cleaned Training Set V2:
['pingo antarctica hale dull slow', 'pingo antarctica hale boring film', 'pingo antarctica hale dull predictable', 'funny film', 'charming beautiful']

Cleaned Test Set V2:
['pingo antarctica hale charming funny']


In [ ]:
# Load the training data
training_data_v2 = cleaned_training_v2

# Create the CountVectorizer (this builds the BoW model)
# default is binary = False, so it counts the number of times a token appears in a sentence
# set binary = True, if we just want to check the presence of a token in a sentence
vectorizer_v2 = CountVectorizer(binary = False)

# Fit and transform the corpus into a BoW matrix
bow_representation_v2 = vectorizer_v2.fit_transform(training_data_v2)

print(f"Bag of Words Representation:\n")
print(bow_representation_v2)

# Notice that the number of elements

Bag of Words Representation:

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 19 stored elements and shape (5, 11)>
  Coords	Values
  (0, 8)	1
  (0, 0)	1
  (0, 7)	1
  (0, 4)	1
  (0, 10)	1
  (1, 8)	1
  (1, 0)	1
  (1, 7)	1
  (1, 2)	1
  (1, 5)	1
  (2, 8)	1
  (2, 0)	1
  (2, 7)	1
  (2, 4)	1
  (2, 9)	1
  (3, 5)	1
  (3, 6)	1
  (4, 3)	1
  (4, 1)	1


In [ ]:
# Perform the training step on our training data
log_prior, log_likelihood_v2, V_v2, C = train_naive_bayes(cleaned_training_v2, labels)

print(f"Log Prior: {log_prior}\n")
print(f"Log Likelihood: {log_likelihood_v2}\n")
print(f"Vocabulary: {V_v2}\n")
print(f"Classes: {C}")

Log Prior: {'Negative': np.float64(-0.5108256237659907), 'Positive': np.float64(-0.916290731874155)}

Log Likelihood: {('antarctica', 'Negative'): np.float64(-1.8718021769015913), ('beautiful', 'Negative'): np.float64(-3.258096538021482), ('boring', 'Negative'): np.float64(-2.5649493574615367), ('charming', 'Negative'): np.float64(-3.258096538021482), ('dull', 'Negative'): np.float64(-2.159484249353372), ('film', 'Negative'): np.float64(-2.5649493574615367), ('funny', 'Negative'): np.float64(-3.258096538021482), ('hale', 'Negative'): np.float64(-1.8718021769015913), ('pingo', 'Negative'): np.float64(-1.8718021769015913), ('predictable', 'Negative'): np.float64(-2.5649493574615367), ('slow', 'Negative'): np.float64(-2.5649493574615367), ('antarctica', 'Positive'): np.float64(-2.70805020110221), ('beautiful', 'Positive'): np.float64(-2.0149030205422647), ('boring', 'Positive'): np.float64(-2.70805020110221), ('charming', 'Positive'): np.float64(-2.0149030205422647), ('dull', 'Positive'):

In [ ]:
argmax, total = test_naive_bayes(cleaned_test_v2[0], log_prior, log_likelihood_v2, C, V_v2)

print(f"Cleaned Test Sentence: {cleaned_test_v2[0]}")
print(f"Predicted Class: {argmax}")
print(f"Total: {total}")

Cleaned Test Sentence: pingo antarctica hale charming funny
Predicted Class: Negative
Total: {'Negative': np.float64(-12.642425230513728), 'Positive': np.float64(-13.070247376265316)}


####

### Wrap it all up in scikit-learn

In [ ]:
# Pipeline tells us that an input will be processed through these steps
# CountVectorizer and then Naive Bayes
# Input must be the pre-processed training data or test data
# Both training and test data will use the same pipeline
model = Pipeline([
    ("bow", CountVectorizer(binary = False)), # Same settings as above
    ("nb", MultinomialNB(alpha = 1)),    # alpha = 1 is Laplace smoothing
])

# Trains the Naive Bayes model on training data of pre-processing version 2
model.fit(training_data_v2, labels)

Pipeline(steps=[('bow', CountVectorizer()), ('nb', MultinomialNB(alpha=1))])

In [ ]:
# Check the train accuracy
print(f"Train accuracy:  {model.score(training_data_v2, labels):.4f}")

Train accuracy:  1.0000


In [ ]:
# The pipeline expects a list as an input so wrap the sentence in a list even if it is just one
# Again, must be the same pre-processed version 2 of test data

# Predicted class
print(f"Prediction: {model.predict(cleaned_test_v2)}")

# Probability of each class (normalized via softmax)
print(f"Prediction Probabilities: {model.predict_proba(cleaned_test_v2)}")

Prediction: ['Negative']
Prediction Probabilities: [[0.6053535 0.3946465]]


In [ ]:
# Access the information of a step
nb = model.named_steps["nb"]
vocab = model.named_steps["bow"].get_feature_names_out()

print("Classes:", nb.classes_)
print("log P(c):", nb.class_log_prior_)

for i, c in enumerate(nb.classes_):
    for w, ll in zip(vocab, nb.feature_log_prob_[i]):
        print(f"log P({w} | {c}) = {ll:.4f}   (scratch: {log_likelihood_v2[(w, c)]:.4f})")

Classes: ['Negative' 'Positive']
log P(c): [-0.51082562 -0.91629073]
log P(antarctica | Negative) = -1.8718   (scratch: -1.8718)
log P(beautiful | Negative) = -3.2581   (scratch: -3.2581)
log P(boring | Negative) = -2.5649   (scratch: -2.5649)
log P(charming | Negative) = -3.2581   (scratch: -3.2581)
log P(dull | Negative) = -2.1595   (scratch: -2.1595)
log P(film | Negative) = -2.5649   (scratch: -2.5649)
log P(funny | Negative) = -3.2581   (scratch: -3.2581)
log P(hale | Negative) = -1.8718   (scratch: -1.8718)
log P(pingo | Negative) = -1.8718   (scratch: -1.8718)
log P(predictable | Negative) = -2.5649   (scratch: -2.5649)
log P(slow | Negative) = -2.5649   (scratch: -2.5649)
log P(antarctica | Positive) = -2.7081   (scratch: -2.7081)
log P(beautiful | Positive) = -2.0149   (scratch: -2.0149)
log P(boring | Positive) = -2.7081   (scratch: -2.7081)
log P(charming | Positive) = -2.0149   (scratch: -2.0149)
log P(dull | Positive) = -2.7081   (scratch: -2.7081)
log P(film | Positive) =

In [ ]:
# Sorted alphabetically by labels
# So, it's "Negative" and then "Positive" log likelihood values of each word (also sorted alphabetically)
nb.feature_log_prob_

array([[-1.87180218, -3.25809654, -2.56494936, -3.25809654, -2.15948425,
        -2.56494936, -3.25809654, -1.87180218, -1.87180218, -2.56494936,
        -2.56494936],
       [-2.7080502 , -2.01490302, -2.7080502 , -2.01490302, -2.7080502 ,
        -2.01490302, -2.01490302, -2.7080502 , -2.7080502 , -2.7080502 ,
        -2.7080502 ]])

In [ ]:
# Row index of each class in feature_log_prob_
pos = list(nb.classes_).index("Positive")
neg = list(nb.classes_).index("Negative")

print(f"Positive index: {pos}")
print(f"Negative index: {neg}")

Positive index: 1
Negative index: 0


In [ ]:
# Log-likelihood ratio per word: > 0 pushes toward positive, < 0 toward negative
word_scores = pd.DataFrame({
    "word": vocab,
    "log P(w|pos)": nb.feature_log_prob_[pos],
    "log P(w|neg)": nb.feature_log_prob_[neg],
})

# Log-likelihood ratio is log P(w | pos) / log P(w | neg)
# Log properties lets it become a difference of the two
word_scores["LLR (pos - neg)"] = word_scores["log P(w|pos)"] - word_scores["log P(w|neg)"]
word_scores["direction"] = word_scores["LLR (pos - neg)"].apply(
    lambda x: "positive" if x > 0 else ("negative" if x < 0 else "neutral"))

# Sort sccores by most impact towards a sentiment
word_scores = word_scores.sort_values("LLR (pos - neg)", ascending = False)

print("Words pushing toward 'Positive':")
print(word_scores.head(5).to_string(index = False))
print("\nWords pushing toward 'Negative':")
print(word_scores.tail(5).iloc[::-1].to_string(index = False)) # The [::-1] is to flip the order of the tail to make it show from most negative to least negative

Words pushing toward 'Positive':
     word  log P(w|pos)  log P(w|neg)  LLR (pos - neg) direction
beautiful     -2.014903     -3.258097         1.243194  positive
    funny     -2.014903     -3.258097         1.243194  positive
 charming     -2.014903     -3.258097         1.243194  positive
     film     -2.014903     -2.564949         0.550046  positive
   boring     -2.708050     -2.564949        -0.143101  negative

Words pushing toward 'Negative':
      word  log P(w|pos)  log P(w|neg)  LLR (pos - neg) direction
      hale      -2.70805     -1.871802        -0.836248  negative
     pingo      -2.70805     -1.871802        -0.836248  negative
antarctica      -2.70805     -1.871802        -0.836248  negative
      dull      -2.70805     -2.159484        -0.548566  negative
      slow      -2.70805     -2.564949        -0.143101  negative


### Airlines Example

Two files have been prepared for you from the airline dataset:

- `airline_train_600.csv`: 600 tweets for training (200 negative, 200 neutral, 200 positive)
- `airline_test_6.csv`: 6 tweets for testing (2 of each label)

Each file has two columns: `airline_sentiment` (the label) and `text` (the tweet).

Do the following:

1. Load both files into dataframes with `pd.read_csv`.
2. **View** some of the tweets before doing anything else. Print a few from each label and read them closely.
3. Clean the text, turn it into word counts with `CountVectorizer`, and train a `MultinomialNB` model on the training set.
4. Predict the labels of the 6 test tweets and compare them with the true labels.

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
# Load the train and test splits
train_df = pd.read_csv("airline_train_900.csv")
test_df = pd.read_csv("airline_test_60.csv")
print(f"Train: {len(train_df)} tweets  {train_df['airline_sentiment'].value_counts().to_dict()}") # Gets the count of each distinct row
print(f"Test:  {len(test_df)} tweets  {test_df['airline_sentiment'].value_counts().to_dict()}\n")

Train: 900 tweets  {'negative': 300, 'neutral': 300, 'positive': 300}
Test:  60 tweets  {'negative': 20, 'neutral': 20, 'positive': 20}



In [ ]:
# View two tweets per label, random state makes it reproducible per run
for sentiment in ["negative", "neutral", "positive"]:
    for _, row in train_df[train_df["airline_sentiment"] == sentiment].sample(2, random_state = 42).iterrows():
        print(f"Label: {row['airline_sentiment']}")
        print(f"Text:  {row['text']}\n")

Label: negative
Text:  @JetBlue supervisor humiliated us and was uncompromising. We were completely blindsided!

Label: negative
Text:  @JetBlue last flight was Cancelled Flightled, then it was 2 hours to change a light bulb, now sitting here waiting for maintenance. Unreal.

Label: neutral
Text:  @usairways sadly they've tried that and airport agents…no flights so they're stranded at PHL. No chance of a hotel for them?

Label: neutral
Text:  “@JetBlue: Our fleet's on fleek. http://t.co/XX5QSCjlL1” @BrandsSayingBae #Freberg15

Label: positive
Text:  @USAirways finally rectified my flight situation! Thanks again

Label: positive
Text:  Thank you @united for your prompt assistance.



Almost every tweet starts with a handle like `@united` or `@SouthwestAir`, and many contain a link like `http://t.co/abc123`.

If you want to remove these, you can remove both with `re.sub(pattern, replacement, text)` from Python's `re` module.
- **Handles:** the pattern `r"@\w+"` means "an `@` followed by one or more letters, digits or underscores."
    - For example, `re.sub(r"@\w+", " ", "@united my bag is lost")` → `"  my bag is lost"`
- **Links:** the pattern `r"http\S+"` means "`http` followed by everything up to the next space."
    - For example, `re.sub(r"http\S+", " ", "great crew http://t.co/abc")` → `"great crew  "`
- After performing these steps, there might be double spaces or leading and trailing spaces, make sure to remove those as well.

In [ ]:
# Remove @mentions and links, then lowercase
def clean(text):
    text = re.sub(r"@\w+", " ", text)        # @united, @SouthwestAir, ...
    text = re.sub(r"http\S+", " ", text)     # http://t.co/...
    text = re.sub(r"\s+", " ", text)         # Collapse leftover spaces
    return text.strip().lower()              # Removes leading and trailing spaces then lowercases

train_df["clean_text"] = train_df["text"].apply(clean)
test_df["clean_text"] = test_df["text"].apply(clean)

# Compare before and after
for _, row in train_df.head(3).iterrows():
    print(f"Before: {row['text']}")
    print(f"After:  {row['clean_text']}\n")

Before: @united, there is no good word for describing how angry and pissed off of a customer you have made me.
After:  , there is no good word for describing how angry and pissed off of a customer you have made me.

Before: @united @perfectomobile huh?  I have sent 16 crash reports in Feb alone.  8 in one day.  Only app I have that consistently crashes.
After:  huh? i have sent 16 crash reports in feb alone. 8 in one day. only app i have that consistently crashes.

Before: @united hi JH. Yes please. Since I lost almost a whole day on the way out, could you help me change my return for one day Late Flightr?
After:  hi jh. yes please. since i lost almost a whole day on the way out, could you help me change my return for one day late flightr?



In [ ]:
# Pipeline: bag of words -> Naive Bayes
model = Pipeline([
    ("bow", CountVectorizer(binary = False, stop_words = "english")),
    ("nb", MultinomialNB(alpha = 1)),
])

# Train on the cleaned tweets
model.fit(train_df["clean_text"], train_df["airline_sentiment"])

print(f"Train accuracy:  {model.score(train_df['clean_text'], train_df['airline_sentiment']):.4f}")

Train accuracy:  0.9311


In [ ]:
# Most indicative words per class: log P(word | class) minus the average over the other classes
# One class compared against the rest, the rest is averaged
vocab = model.named_steps["bow"].get_feature_names_out()
nb = model.named_steps["nb"]

print(f"Vocabulary size: {len(vocab)}")
log_probs = nb.feature_log_prob_
for i, label in enumerate(nb.classes_):
    others = log_probs[[j for j in range(len(nb.classes_)) if j != i]].mean(axis = 0)
    top = (log_probs[i] - others).argsort()[::-1][:10]   # Returns the indices of the values, ordered from largest to smallest (via [::-1]) of the top 10
    print(f"Top {label} words: {', '.join(vocab[top])}")

Vocabulary size: 2401
Top negative words: hours, hold, bags, line, delayed, lost, waiting, minutes, flightled, hour
Top neutral words: question, fleek, fleet, ceo, atlanta, possible, cabin, rt, dfw, nashville
Top positive words: amazing, thank, thx, great, quick, excellent, easy, fast, wonderful, makes


In [ ]:
# Inputs and true labels for the test set
X_test = test_df["clean_text"]
y_test = test_df["airline_sentiment"]

# Predict the label and the probability of each class
test_df["predicted"] = model.predict(test_df["clean_text"])
probs = model.predict_proba(test_df["clean_text"])

for i, label in enumerate(nb.classes_):
    test_df[f"P({label})"] = probs[:, i].round(4)

# Show 6 example predictions (2 per true label); the summary below still uses all test tweets
examples = test_df.groupby("airline_sentiment", group_keys = False).head(2)

print(f"Predictions on the test set (showing {len(examples)} of {len(test_df)})\n")
for _, row in examples.iterrows():
    mark = "correct" if row["predicted"] == row["airline_sentiment"] else "incorrect"
    prob_str = ", ".join(f"{c} = {row[f'P({c})']:.3f}" for c in nb.classes_)
    print(f"Text:      {row['text']}")
    print(f"True:      {row['airline_sentiment']}")
    print(f"Predicted: '{row['predicted']}' is {mark}")
    print(f"Probs:     {prob_str}\n")

Predictions on the test set (showing 6 of 60)

Text:      @JetBlue before departure, while people who are delayed get up to $200 to compensate their time..
True:      negative
Predicted: 'negative' is correct
Probs:     negative = 0.625, neutral = 0.095, positive = 0.280

Text:      @united yes please! I am newly married and trying to update my last name on a preexisting international flight! It seems so easy, but...
True:      negative
Predicted: 'neutral' is incorrect
Probs:     negative = 0.321, neutral = 0.412, positive = 0.267

Text:      @JetBlue? | RT @WMCActionNews5: New airline expected to make its way to MEM http://t.co/vO3eQsEPqH
True:      neutral
Predicted: 'neutral' is correct
Probs:     negative = 0.023, neutral = 0.565, positive = 0.412

Text:      @USAirways got transf. to Dividend Miles...need to change flight.  Grandma just passed, need to change ticket ASAP. Update on wait time?
True:      neutral
Predicted: 'negative' is incorrect
Probs:     negative = 0.547, neutr

In [ ]:
# Summary over the full test set
correct = (y_test == test_df["predicted"]).sum()
print(f"Test accuracy: {accuracy_score(y_test, test_df['predicted']):.4f} ({correct} of {len(test_df)} correct)\n")

# Create the confusion matrix
print(pd.DataFrame(confusion_matrix(y_test, test_df["predicted"], labels = nb.classes_),
                   index = [f"true {c}" for c in nb.classes_],
                   columns = [f"pred {c}" for c in nb.classes_]))

Test accuracy: 0.6333 (38 of 60 correct)

               pred negative  pred neutral  pred positive
true negative             16             2              2
true neutral              10             8              2
true positive              2             4             14


In [ ]:
print(classification_report(y_test, test_df["predicted"], labels = nb.classes_, digits = 4))

# Precision - when the model says negative, can I trust it?
# It looks at the model’s predictions
# Recall - of all the tweets that really are negative, how many did the model find?
# It looks at the true labels

              precision    recall  f1-score   support

    negative     0.5714    0.8000    0.6667        20
     neutral     0.5714    0.4000    0.4706        20
    positive     0.7778    0.7000    0.7368        20

    accuracy                         0.6333        60
   macro avg     0.6402    0.6333    0.6247        60
weighted avg     0.6402    0.6333    0.6247        60

